<a href="https://colab.research.google.com/github/kreaja/TrialRepositoryJustForMe/blob/main/hw05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW05: Neural Architectures (RNN → Transformer)
## CIS 5300: Natural Language Processing | Modules 5 and 6 | 100 Points

---

### Overview

In this assignment, you will build two character-level language models on the Tiny Shakespeare dataset: first a simple **RNN baseline** using PyTorch's built-in `nn.RNN`, then a complete **decoder-only Transformer** from scratch. By implementing both architectures on the same task, you will directly compare how recurrence and self-attention tackle sequence modeling.

You will implement each component of the transformer architecture piece by piece -- character tokenization, causal self-attention, multi-head attention, feedforward layers, residual connections, and layer normalization -- then assemble them into a complete model, train it, and generate text.

This homework connects directly to the Module 05 and Module 06 lectures:
- **RNNs and Sequence Modeling:** Lectures M05.2.6–M05.2.9
- **Transformer Architecture:** Lectures M06.1.1–M06.1.7
- **Neural Text Generation:** Lectures M06.2.0–M06.2.3

**Textbook Reference:** [Chapter 8: Transformers](https://web.stanford.edu/~jurafsky/slp3/8.pdf) of Speech and Language Processing by Jurafsky & Martin.

**Note:** Training the full Transformer takes approximately 15–30 minutes on CPU and ~2 minutes on GPU. GPU is recommended but not required.

### Point Breakdown

| Section | Topic | Points | Type |
|---------|-------|--------|------|
| 0 | Setup | 0 | -- |
| 1 | PyTorch Foundations & RNN Baseline | 15 | Code |
| 2 | Self-Attention | 25 | Code |
| 3 | Transformer Block | 20 | Code |
| 4 | Full Model, Training & Generation | 23 | Code + Writeup |
| 5 | RNN vs Transformer Analysis | 10 | Writeup |
| 6 | Mathematical Analysis | 7 | Writeup |
| **Total** | | **100** | |

### Deliverables

Submit the following to Gradescope:

| File | Description |
|------|-------------|
| `hw05.ipynb` | This notebook with all code cells completed |
| `hw05.py`             | Code from this notebook, downloaded as a .py file     |
| `hw05_writeup.pdf` | Written answers to all questions marked **[WRITEUP]** |

### Recommended Reading

- [Chapter 8: Transformers](https://web.stanford.edu/~jurafsky/slp3/8.pdf) -- Jurafsky & Martin, SLP 3rd Edition
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) -- Alammar, 2018
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) -- Vaswani et al., 2017
- Module 05 and Module 06 Lectures


---
## Section 0: Setup (0 points)

Run the following cells to install packages, load the dataset, and set hyperparameters. **Do not modify these cells** unless you need to adjust the device setting.

In [17]:
# Install required packages (uncomment if needed)
# !pip install torch numpy matplotlib requests

In [18]:
# (What package versions should we use?)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import requests
import time

# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device detection
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cpu':
    print("NOTE: Training the transformer on CPU takes ~15-30 minutes.")
    print("      If you can use a GPU, training drops to ~2-3 minutes.")
    print("      In Colab: Runtime -> Change runtime type -> T4 GPU (free).")

print(f"PyTorch version: {torch.__version__}")
print("All imports successful.")

Using device: cpu
NOTE: Training the transformer on CPU takes ~15-30 minutes.
      If you can use a GPU, training drops to ~2-3 minutes.
      In Colab: Runtime -> Change runtime type -> T4 GPU (free).
PyTorch version: 2.11.0+cpu
All imports successful.


### 0.1 Load the Tiny Shakespeare Dataset

We use the [Tiny Shakespeare](https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt) dataset, a ~1MB text file containing all of Shakespeare's works concatenated together. This is a standard benchmark for small language models.

In [19]:
########## DO NOT MODIFY ##########

# Load Tiny Shakespeare dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
corpus = requests.get(url).text

print(f"Corpus length: {len(corpus):,} characters")
print(f"\nFirst 200 characters:")
print(corpus[:200])

Corpus length: 1,115,394 characters

First 200 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


### 0.2 Hyperparameters

These are reasonable defaults for a small character-level transformer. You do not need to modify these, but you are welcome to experiment after completing the assignment.

In [20]:
########## DO NOT MODIFY ##########

# Model hyperparameters
batch_size = 32        # Number of sequences in each training batch
block_size = 64        # Context window length (number of tokens the model sees at once)
n_embd = 128           # Embedding dimension
n_head = 4             # Number of attention heads in multi-head attention
n_layer = 4            # Number of stacked transformer blocks
dropout_rate = 0.1     # Dropout probability for regularization
learning_rate = 3e-4   # Learning rate for AdamW optimizer

print("Hyperparameters:")
print(f"  batch_size  = {batch_size}")
print(f"  block_size  = {block_size}")
print(f"  n_embd      = {n_embd}")
print(f"  n_head      = {n_head}")
print(f"  n_layer     = {n_layer}")
print(f"  dropout     = {dropout_rate}")
print(f"  lr          = {learning_rate}")
print(f"  head_size   = n_embd // n_head = {n_embd // n_head}")

Hyperparameters:
  batch_size  = 32
  block_size  = 64
  n_embd      = 128
  n_head      = 4
  n_layer     = 4
  dropout     = 0.1
  lr          = 0.0003
  head_size   = n_embd // n_head = 32


---
## Section 1: PyTorch Foundations & RNN Baseline (15 points)

Before we can train a language model, we need to convert raw text into sequences of integers, set up data batching, and establish a baseline. We use a **character-level tokenizer** (each unique character → unique integer ID), then set up next-token batches, and finally build a small **RNN baseline** using PyTorch's `nn.RNN`. This baseline will give us a point of comparison when you build the Transformer in Sections 2–4.

**Lectures:** M05.2.6–M05.2.9 (RNNs), M06.1.1 (Tokenization) | **Textbook:** Chapter 8, Section 8.1


### 1.1 Character Tokenizer (5 points)

Implement a character-level tokenizer with `encode` and `decode` functions.

1. Create `vocab` -- a sorted list of all unique characters in the corpus.
2. Create `stoi` -- a dictionary mapping each character to its integer index.
3. Create `itos` -- a dictionary mapping each integer index back to its character.
4. Implement `encode(string)` that converts a string to a list of integers.
5. Implement `decode(int_list)` that converts a list of integers back to a string.

These should be inverse operations: `decode(encode(s)) == s` for any string `s` composed of characters in the vocabulary.

In [21]:
# Create a sorted list of all unique characters in the corpus
vocab = sorted(list(set(chr for chr in corpus)))
# Will get !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab_size = len(vocab)

# Create character-to-index mapping
stoi = {cr: i for i, cr in enumerate(vocab)}  # They mean its index in the array, not the ASCII value of the char

# Create index-to-character mapping
itos = {i: cr for i, cr in enumerate(vocab)}

# Encode: string -> list of integers
def encode(s):
    """Convert a string to a list of integer token IDs.

    Args:
        s: A string composed of characters in the vocabulary.

    Returns:
        A list of integers, one per character.
    """
    return [stoi[c] for c in s]

# Decode: list of integers -> string
def decode(int_list):
    """Convert a list of integer token IDs back to a string.

    Args:
        int_list: A list of integers representing token IDs.

    Returns:
        A string.
    """
    decoded_string = [itos[i] for i in int_list]
    return "".join(decoded_string)

In [22]:
# Sanity checks for tokenizer

assert vocab is not None and len(vocab) > 0, "vocab should be a non-empty list"
assert vocab == sorted(vocab), "vocab should be sorted"
assert vocab_size == len(vocab), f"vocab_size should equal len(vocab), got {vocab_size} vs {len(vocab)}"
assert len(stoi) == vocab_size, "stoi should have one entry per character"
assert len(itos) == vocab_size, "itos should have one entry per character"

# Round-trip test
test_str = "hi there"
assert decode(encode(test_str)) == test_str, \
    f"encode/decode round-trip failed: got '{decode(encode(test_str))}' instead of '{test_str}'"

# Check specific encoding (Tiny Shakespeare has a known character set)
assert encode("hi there") == [46, 47, 1, 58, 46, 43, 56, 43], \
    f"Unexpected encoding for 'hi there': {encode('hi there')}"

# Full corpus round-trip
assert decode(encode(corpus[:100])) == corpus[:100], "Full corpus round-trip failed"

print(f"Vocabulary size: {vocab_size} characters")
print(f"Characters: {''.join(vocab)}")
print(f"encode('hi there') = {encode('hi there')}")
print(f"decode([46, 47, 1, 58, 46, 43, 56, 43]) = '{decode([46, 47, 1, 58, 46, 43, 56, 43])}'")
print("\nTokenizer sanity checks passed!")

Vocabulary size: 65 characters
Characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
encode('hi there') = [46, 47, 1, 58, 46, 43, 56, 43]
decode([46, 47, 1, 58, 46, 43, 56, 43]) = 'hi there'

Tokenizer sanity checks passed!


### 1.2 Data Batching (5 points)

We need to prepare the data for training. This involves:
1. Encoding the entire corpus as a tensor of integers.
2. Splitting into training (90%) and validation (10%) sets.
3. Implementing a `get_batch` function that samples random mini-batches.

Each training example consists of a pair `(x, y)` where:
- `x` is a sequence of `block_size` tokens (the input context)
- `y` is the same sequence shifted right by one position (the targets)

For example, if the text is `"hello"` and `block_size=4`:
- `x = [h, e, l, l]`
- `y = [e, l, l, o]`

The model learns to predict each next token given the preceding context.

In [23]:
########## DO NOT MODIFY ##########

# Encode the entire corpus as a tensor
data = torch.tensor(encode(corpus), dtype=torch.long)
print(f"Encoded corpus: {data.shape[0]:,} tokens")

# Train/val split: first 90% for training, last 10% for validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Training tokens:   {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")
# Outputs "Encoded corpus: 1,115,394 tokens"

Encoded corpus: 1,115,394 tokens
Training tokens:   1,003,854
Validation tokens: 111,540


In [24]:
if torch.cuda.is_available():
    print("cuda is available")
elif torch.backends.mps.is_available():
    print("mps is available")
else:
    print("Only cpu is available")

def get_batch(split):
    """Sample a random mini-batch of (input, target) pairs.

    Args:
        split: 'train' or 'val' -- which dataset to sample from.

    Returns:
        x: Tensor of shape (batch_size, block_size) -- input token sequences.
        y: Tensor of shape (batch_size, block_size) -- target token sequences (shifted by 1).

    Steps:
        1. Select train_data or val_data based on the split argument.
        2. Sample batch_size random starting indices.
           Each index i should satisfy: 0 <= i <= len(data) - block_size - 1
           (we need block_size + 1 tokens starting from i to create both x and y).
        3. For each starting index i:
           - x[j] = data[i : i + block_size]
           - y[j] = data[i+1 : i + block_size + 1]
        4. Stack into tensors and move to device.

    Hint: Use torch.randint for sampling and torch.stack for batching.
    """
    if split == 'train':
        data_to_use = train_data
    elif split == 'val':
        data_to_use = val_data
    start_indices = torch.randint(0, len(data_to_use) - block_size - 1, (batch_size,))
                                               # Note - a batch_size x 1 column vector

    x = torch.stack([data_to_use[i : i + block_size] for i in start_indices])
    # Each training example consists of a pair (x, y) where:
      # x is a sequence of block_size tokens (the input context)
      # y is the same sequence shifted right by one position (the targets)
    y = torch.stack([data_to_use[i + 1 : i + block_size + 1] for i in start_indices])
    return x.to(device), y.to(device)


Only cpu is available


In [25]:
# Sanity checks for get_batch

xb, yb = get_batch('train')

assert xb.shape == (batch_size, block_size), \
    f"x shape should be ({batch_size}, {block_size}), got {xb.shape}"
assert yb.shape == (batch_size, block_size), \
    f"y shape should be ({batch_size}, {block_size}), got {yb.shape}"
assert xb.dtype == torch.long, f"x should be LongTensor, got {xb.dtype}"
assert yb.dtype == torch.long, f"y should be LongTensor, got {yb.dtype}"

# Verify that y is x shifted by 1: for any batch element, y should start
# where x's second token is
# (This checks that the shift relationship holds)
assert torch.all(xb[:, 1:] == yb[:, :-1]), \
    "y should be x shifted right by 1 position"

# Check val split works too
xv, yv = get_batch('val')
assert xv.shape == (batch_size, block_size), "val batch x has wrong shape"

print(f"Train batch x shape: {xb.shape}")
print(f"Train batch y shape: {yb.shape}")
print(f"\nFirst training example (first 20 tokens):")
print(f"  x: {xb[0, :20].tolist()}")
print(f"  y: {yb[0, :20].tolist()}")
print(f"  x decoded: '{decode(xb[0, :20].tolist())}'")
print(f"  y decoded: '{decode(yb[0, :20].tolist())}'")
print("\nData batching sanity checks passed!")

Train batch x shape: torch.Size([32, 64])
Train batch y shape: torch.Size([32, 64])

First training example (first 20 tokens):
  x: [42, 6, 1, 50, 47, 54, 57, 6, 1, 27, 1, 63, 53, 59, 0, 32, 46, 43, 1, 42]
  y: [6, 1, 50, 47, 54, 57, 6, 1, 27, 1, 63, 53, 59, 0, 32, 46, 43, 1, 42, 53]
  x decoded: 'd, lips, O you
The d'
  y decoded: ', lips, O you
The do'

Data batching sanity checks passed!


### 1.3 RNN Baseline LM (5 points)

We will set up an **RNN baseline** using PyTorch's built-in `nn.RNN`. This gives us a direct comparison point against the Transformer you will build in Sections 2–4: same data, same task (next-character prediction), different architecture.

Implement a small character-level RNN that:
1. Embeds input character IDs through `nn.Embedding`.
2. Passes the embeddings through a single layer of `nn.RNN`.
3. Projects the per-position hidden states to vocabulary logits.

Make `forward()` accept an optional `targets` argument and return `(logits, loss)`, matching the interface that the `TransformerLM` will use in Section 4. This way the same `get_batch()` infrastructure works for both models, making the comparison fair.

**Notation used throughout:** `B` = batch size (number of sequences processed simultaneously), `T` = sequence length (number of tokens per sequence). Input tensors have shape `(B, T)`, and the model produces logits of shape `(B, T, vocab_size)`.

In [26]:
class CharRNN_LM(nn.Module):
    """A character-level RNN language model using nn.RNN.

    Architecture:
        nn.Embedding(vocab_size, n_embd)
        -> nn.RNN(input_size=n_embd, hidden_size=n_embd, batch_first=True)
        -> nn.Linear(n_embd, vocab_size)

    The forward signature mirrors the TransformerLM you will build in Section 4:
    given idx of shape (B, T), return logits of shape (B, T, vocab_size).
    If targets are provided, also return the cross-entropy loss.
    """

    def __init__(self, vocab_size, n_embd):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.rnn = nn.RNN(input_size=n_embd, hidden_size=n_embd, batch_first=True)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        """Forward pass.

        Args:
            idx: Token indices, shape (B, T).
            targets: Optional target indices, shape (B, T).

        Returns:
            logits: Shape (B, T, vocab_size).
            loss: Scalar cross-entropy loss if targets is provided, else None.

        Hint: Use F.cross_entropy on logits reshaped to (B*T, vocab_size) and
              targets reshaped to (B*T,).
        """
        # F is an abbreviation for torch.nn.functional, a module that contains
        # **stateless** operations for building neural networks.
        x = self.token_emb(idx)   # <- the embedded input.
        out, h_n = self.rnn(x)
        logits = self.lm_head(out)   # B is the batch size
        if targets is None:      # T is the sequence length (number of chars/sequence)
            loss = None       # C ("channels"), here, refers to the vocab size.
        else:    # out is the hidden state at every time step, having shape (B, T, n_embd). out[b, t] summarizes
            B, T, C = logits.shape    #  everything the RNN has read in sequence `b` up through position `t`.
            lo_view = logits.view(B * T, C)  # Calling lm_head() applies a learned linear map, out @ W.T + b,
            targets = targets.view(B * T)  # to the last dimension only, leaving the leading dimensions alone.
            loss = F.cross_entropy(lo_view, targets)
        return (logits, loss)

In [27]:
########## DO NOT MODIFY ##########

# Sanity check: forward pass produces logits of correct shape and a scalar loss
import torch.nn.functional as F
torch.manual_seed(0)
test_rnn = CharRNN_LM(vocab_size, n_embd).to(device)
xb_test, yb_test = get_batch('train')
logits_rnn, loss_rnn = test_rnn(xb_test, yb_test)

assert logits_rnn.shape == (batch_size, block_size, vocab_size), \
    f"Expected logits shape ({batch_size}, {block_size}, {vocab_size}), got {tuple(logits_rnn.shape)}"
assert loss_rnn is not None and loss_rnn.dim() == 0, "loss should be a scalar tensor when targets are provided"
print(f"CharRNN_LM forward OK. Logits shape: {tuple(logits_rnn.shape)}, initial loss: {loss_rnn.item():.4f}")
print(f"Random-baseline loss = ln(vocab_size) = {torch.log(torch.tensor(float(vocab_size))).item():.4f}")


CharRNN_LM forward OK. Logits shape: (32, 64, 65), initial loss: 4.1913
Random-baseline loss = ln(vocab_size) = 4.1744


In [28]:
# Colab: Run takes about 8 s on a T4 (GPU), but 34 s using CPU.

if __name__ == '__main__':
    ########## DO NOT MODIFY ##########

    # Brief training of the RNN baseline for comparison with the Transformer (later in S4).
    # We run only 1000 iterations -- enough to see meaningful learning but not full training.

    torch.manual_seed(0)
    rnn_baseline = CharRNN_LM(vocab_size, n_embd).to(device)
    optimizer = torch.optim.AdamW(rnn_baseline.parameters(), lr=learning_rate)
    rnn_train_losses = []
    rnn_val_losses = []
    n_iters_baseline = 1000
    eval_interval_baseline = 100
    print("Training RNN baseline...")
    for step in range(n_iters_baseline):
        xb, yb = get_batch('train')
        _, loss = rnn_baseline(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if step % eval_interval_baseline == 0 or step == n_iters_baseline - 1:
            with torch.no_grad():
                rnn_baseline.eval()
                xb_v, yb_v = get_batch('val')
                _, val_loss = rnn_baseline(xb_v, yb_v)
                rnn_baseline.train()
            rnn_train_losses.append(loss.item())
            rnn_val_losses.append(val_loss.item())
            print(f"  step {step:4d} | train loss {loss.item():.4f} | val loss {val_loss.item():.4f}")

    final_rnn_val_loss = rnn_val_losses[-1]
    print(f"\nFinal RNN baseline val loss after {n_iters_baseline} steps: {final_rnn_val_loss:.4f}")
    print("Keep this number in mind -- you'll compare it to the Transformer's val loss in Section 5.")

# Final RNN baseline val loss after 1000 steps: 2.0558   Keep this number in mind.

Training RNN baseline...
  step    0 | train loss 4.1913 | val loss 4.1572
  step  100 | train loss 2.6778 | val loss 2.7872
  step  200 | train loss 2.4549 | val loss 2.4546
  step  300 | train loss 2.3616 | val loss 2.3688
  step  400 | train loss 2.2406 | val loss 2.3100
  step  500 | train loss 2.2241 | val loss 2.2175
  step  600 | train loss 2.1715 | val loss 2.1606
  step  700 | train loss 2.0705 | val loss 2.1553
  step  800 | train loss 2.0829 | val loss 2.1065
  step  900 | train loss 2.0138 | val loss 2.0483
  step  999 | train loss 2.0237 | val loss 2.0558

Final RNN baseline val loss after 1000 steps: 2.0558
Keep this number in mind -- you'll compare it to the Transformer's val loss in Section 5.


---
## Section 2: Self-Attention (25 points)

Self-attention is the core mechanism of the transformer. It allows each token in the sequence to "attend" to every other token, computing a weighted combination of their representations. In a **decoder-only** transformer (used for language modeling), we apply a **causal mask** so that each token can only attend to itself and preceding tokens -- never to future tokens.

The attention formula (from Vaswani et al., 2017):

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

where $Q$ (queries), $K$ (keys), and $V$ (values) are linear projections of the input, and $d_k$ is the dimension of each attention head.

**Lectures:** M06.1.2--M06.1.4 | **Textbook:** Chapter 8, Sections 8.2--8.3

### 2.1 Causal Mask (5 points)

For autoregressive language modeling, we must prevent each position from attending to future positions. We do this by creating a **lower-triangular mask** and setting the upper triangle to $-\infty$ before the softmax. After softmax, $e^{-\infty} = 0$, so future positions receive zero attention weight.

The masked attention matrix for a sequence of length 4 looks like:

$$\begin{bmatrix}
q_1 \cdot k_1 & -\infty & -\infty & -\infty \\
q_2 \cdot k_1 & q_2 \cdot k_2 & -\infty & -\infty \\
q_3 \cdot k_1 & q_3 \cdot k_2 & q_3 \cdot k_3 & -\infty \\
q_4 \cdot k_1 & q_4 \cdot k_2 & q_4 \cdot k_3 & q_4 \cdot k_4
\end{bmatrix}$$

Implement `get_causal_mask(seq_len)` that returns a matrix of shape `(seq_len, seq_len)` with 0 in the lower triangle (including diagonal) and $-\infty$ in the upper triangle.

**Hint:** Use `torch.tril` and `torch.masked_fill` (or `float('-inf')`).

**Applying the mask** means *adding* it to the raw attention scores before softmax: `scores = scores + mask`. Because $-\infty + \text{anything} = -\infty$, and $\text{softmax}(-\infty) = 0$, future positions get exactly zero attention weight.

In [29]:
def get_causal_mask(seq_len):
    """Create a causal (lower-triangular) attention mask.

    Args:
        seq_len: Integer, the sequence length.

    Returns:
        A tensor of shape (seq_len, seq_len) where:
        - Lower triangle (including diagonal) contains 0.0
        - Upper triangle contains float('-inf')

    Example for seq_len=3:
        [[  0, -inf, -inf],
         [  0,    0, -inf],
         [  0,    0,    0]]
    """
    mask = torch.zeros(seq_len, seq_len)
    upper = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
    mask = mask.masked_fill(upper, float('-inf'))
    return mask

In [30]:
# Sanity checks for causal mask

mask = get_causal_mask(4)

assert mask.shape == (4, 4), f"Mask shape should be (4, 4), got {mask.shape}"

# Check lower triangle is 0
for i in range(4):
    for j in range(i + 1):
        assert mask[i, j] == 0.0, f"Lower triangle at ({i},{j}) should be 0, got {mask[i,j]}"

# Check upper triangle is -inf
for i in range(4):
    for j in range(i + 1, 4):
        assert mask[i, j] == float('-inf'), \
            f"Upper triangle at ({i},{j}) should be -inf, got {mask[i,j]}"

# Check that applying softmax to masked scores gives zero for future positions
scores = torch.randn(4, 4).to(device)  # send to device for consistency
masked_scores = scores + mask.to(device)  # mask must be on same device as scores
attn_weights = F.softmax(masked_scores, dim=-1)
for i in range(4):
    for j in range(i + 1, 4):
        assert attn_weights[i, j].item() < 1e-6, \
            f"After softmax, future position ({i},{j}) should be ~0, got {attn_weights[i,j].item()}"

print("Causal mask:")
print(mask)
print(f"\nAfter softmax (row 2): {attn_weights[2].tolist()}")
print("  (Note: only positions 0, 1, 2 have nonzero weight)")
print("\nCausal mask sanity checks passed!")

Causal mask:
tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])

After softmax (row 2): [0.5419925451278687, 0.262657105922699, 0.19535037875175476, 0.0]
  (Note: only positions 0, 1, 2 have nonzero weight)

Causal mask sanity checks passed!


### 2.2 Single-Head Self-Attention (10 points)

Implement a single attention head. Each head independently computes:

1. **Project** the input $x$ into queries ($Q$), keys ($K$), and values ($V$) using separate linear layers.
2. **Compute attention scores:** $\text{scores} = QK^\top / \sqrt{d_k}$  
   *(Here $^\top$ denotes **matrix transpose** — not the variable `T` for sequence length.  
   In PyTorch: `K.transpose(-2, -1)` or the shorthand `K.mT`.)*
3. **Apply causal mask** to prevent attending to future tokens.
4. **Apply softmax** to get attention weights (each row sums to 1).
5. **Apply dropout** to the attention weights for regularization.
6. **Multiply** attention weights by values: $\text{output} = \text{weights} \times V$

The head operates on vectors of size `head_size` (not `n_embd`), since in multi-head attention each head works on a slice of the full embedding.

In [31]:
class Head(nn.Module):
    """A single head of self-attention.

    Each head projects the input into Q, K, V of size head_size,
    computes scaled dot-product attention with causal masking,
    and returns the weighted combination of values.
    """

    def __init__(self, n_embd, head_size, dropout_rate):
        """Initialize a single attention head.

        Args:
            n_embd: Input embedding dimension.
            head_size: Dimension of Q, K, V for this head.
            dropout_rate: Dropout probability for attention weights.

        Create:
            self.query: nn.Linear(n_embd, head_size, bias=False)
            self.key:   nn.Linear(n_embd, head_size, bias=False)
            self.value: nn.Linear(n_embd, head_size, bias=False)
            self.dropout: nn.Dropout(dropout_rate)
        """
        super().__init__()

        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key =  nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        """Forward pass for a single attention head.

        Args:
            x: Input tensor of shape (B, T, n_embd)
               B = batch size, T = sequence length

        Returns:
            Output tensor of shape (B, T, head_size)
            self.attn_weights: Store the attention weights (after softmax,
                before dropout) as self.attn_weights for visualization later.
                Shape: (B, T, T)

        Steps:
            1. Compute Q, K, V by passing x through the linear layers
            2. Compute attention scores: Q @ K^T / sqrt(head_size)
            3. Apply causal mask (use get_causal_mask with seq_len = T)
           Device hint: get_causal_mask() returns a CPU tensor. Move it to
           the same device as scores with .to(scores.device) before adding,
           otherwise you will get a device mismatch error on CUDA.
            4. Apply softmax along the last dimension
            5. Store attention weights as self.attn_weights (for visualization)
            6. Apply dropout to attention weights
            7. Multiply attention weights by V
            8. Return the output
        """

        B, T, C = x.shape
        print(f"B: {B}.   T: {T}.   C: {C}.")  # debug ~ DWN
        Q = self.query(x)
        print(Q.shape)  # debug ~ DWN
        K = self.key(x)
        print(K.shape)  # debug ~ DWN
        V = self.value(x)
        head_size = K.size(-1)
        attention_scores = Q @ K.mT / head_size**0.5
        cpu_tensor = get_causal_mask(seq_len=T)
        attention_weights = attention_scores.detach().cpu()  # ?? ?
        attention_scores = attention_scores + cpu_tensor.to(attention_scores.device)
        attention_scores = torch.nn.functional.softmax(attention_scores, dim=-1)
        self.attn_weights = attention_weights
        attention_weights = self.dropout(attention_weights)
        output = attention_weights @ V
        return output

In [32]:
# Sanity checks for Head

head_size_test = 32
test_head = Head(n_embd, head_size_test, dropout_rate)
test_head.eval()  # Turn off dropout for testing

# Check layer dimensions
assert test_head.query.in_features == n_embd, "Query input size should be n_embd"
assert test_head.query.out_features == head_size_test, "Query output size should be head_size"
assert test_head.key.in_features == n_embd, "Key input size should be n_embd"
assert test_head.value.in_features == n_embd, "Value input size should be n_embd"

# Check forward pass
B_test, T_test = 2, 8
x_test = torch.randn(B_test, T_test, n_embd)
with torch.no_grad():
    out_test = test_head(x_test)

assert out_test.shape == (B_test, T_test, head_size_test), \
    f"Output shape should be ({B_test}, {T_test}, {head_size_test}), got {out_test.shape}"

# Check that attention weights were stored and have correct shape
assert hasattr(test_head, 'attn_weights'), \
    "Head should store attention weights as self.attn_weights"
assert test_head.attn_weights.shape == (B_test, T_test, T_test), \
    f"Attention weights shape should be ({B_test}, {T_test}, {T_test}), got {test_head.attn_weights.shape}"

# Check that attention weights sum to 1 along last dimension
attn_sums = test_head.attn_weights.sum(dim=-1)
assert torch.allclose(attn_sums, torch.ones_like(attn_sums), atol=1e-5), \
    "Attention weights should sum to 1 along the last dimension"

# Check causality: upper triangle of attention weights should be ~0
for i in range(T_test):
    for j in range(i + 1, T_test):
        assert test_head.attn_weights[0, i, j].item() < 1e-6, \
            f"Position {i} should not attend to future position {j}"

print(f"Head output shape: {out_test.shape}")
print(f"Attention weights shape: {test_head.attn_weights.shape}")
print(f"Attention weights row 0 sums to: {test_head.attn_weights[0, 0].sum().item():.6f}")
print("\nSingle-head attention sanity checks passed!")

NameError: name 'head_size' is not defined

### 2.3 Multi-Head Attention (10 points)

Multi-head attention runs several attention heads **in parallel**, each with its own Q, K, V projections of size `head_size = n_embd // n_head`. The outputs of all heads are concatenated and then projected back to `n_embd` dimensions with a linear layer.

$$\text{MultiHead}(x) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$

where each $\text{head}_i = \text{Attention}(xW^Q_i, xW^K_i, xW^V_i)$.

This allows the model to attend to information from different representation subspaces at different positions. For example, one head might learn to attend to the previous word, while another attends to syntactic structure.

**Lectures:** M06.1.4 | **Textbook:** Chapter 8, Section 8.3

**Hint:** Use `torch.cat(list_of_tensors, dim=-1)` to concatenate the head outputs along the last dimension. ([torch.cat docs](https://pytorch.org/docs/stable/generated/torch.cat.html))

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head self-attention: runs multiple Head modules in parallel.

    Each head operates on a slice of the embedding dimension.
    Outputs are concatenated and projected back to n_embd.
    """

    def __init__(self, n_embd, n_head, dropout_rate):
        """Initialize multi-head attention.

        Args:
            n_embd: Full embedding dimension.
            n_head: Number of attention heads.
            dropout_rate: Dropout probability.

        Create:
            self.heads: nn.ModuleList of n_head Head modules,
                        each with head_size = n_embd // n_head
            self.proj: nn.Linear(n_embd, n_embd) -- output projection
            self.dropout: nn.Dropout(dropout_rate)
        """
        super().__init__()

        headsize = n_embd // n_head
        self.heads = nn.ModuleList(n_head, head_size=headsize)
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        """Forward pass for multi-head attention.

        Args:
            x: Input tensor of shape (B, T, n_embd)

        Returns:
            Output tensor of shape (B, T, n_embd)

        Steps:
            1. Run each head on x (each produces shape (B, T, head_size))
            2. Concatenate all head outputs along the last dimension using torch.cat(..., dim=-1)
               Result shape: (B, T, n_head * head_size) = (B, T, n_embd)
            3. Pass through the output projection linear layer
            4. Apply dropout
            5. Return the result
        """
        ## YOUR CODE HERE
        pass

In [ ]:
# Sanity checks for MultiHeadAttention

test_mha = MultiHeadAttention(n_embd, n_head, dropout_rate)
test_mha.eval()

# Check number of heads
assert len(test_mha.heads) == n_head, \
    f"Should have {n_head} heads, got {len(test_mha.heads)}"

# Check output projection
assert test_mha.proj.in_features == n_embd, "Projection input should be n_embd"
assert test_mha.proj.out_features == n_embd, "Projection output should be n_embd"

# Check forward pass: output shape should match input shape
B_test, T_test = 2, 8
x_test = torch.randn(B_test, T_test, n_embd)
with torch.no_grad():
    out_test = test_mha(x_test)

assert out_test.shape == (B_test, T_test, n_embd), \
    f"Output shape should be ({B_test}, {T_test}, {n_embd}), got {out_test.shape}"

# Each head should have its own attention weights
for i, head in enumerate(test_mha.heads):
    assert hasattr(head, 'attn_weights'), f"Head {i} should store attn_weights"

print(f"Number of heads: {len(test_mha.heads)}")
print(f"Head size: {n_embd // n_head}")
print(f"Input shape:  {x_test.shape}")
print(f"Output shape: {out_test.shape}")
print("\nMulti-head attention sanity checks passed!")

---
## Section 3: Transformer Block (20 points)

A transformer block combines multi-head attention with a feedforward network, using **residual connections** and **layer normalization** around each sublayer. Modern transformers use **pre-norm** architecture (LayerNorm before the sublayer, not after), which improves training stability.

The block computation is:
$$x = x + \text{MultiHeadAttention}(\text{LayerNorm}(x))$$
$$x = x + \text{FeedForward}(\text{LayerNorm}(x))$$

**Lectures:** M06.1.5--M06.1.6 | **Textbook:** Chapter 8, Sections 8.3--8.4

### 3.1 Feed-Forward Network (5 points)

Each transformer block contains a position-wise feedforward network (FFN). This is a simple two-layer MLP applied independently to each token position:

$$\text{FFN}(x) = \text{Linear}_2(\text{ReLU}(\text{Linear}_1(x)))$$

The inner dimension is typically 4x the embedding dimension, creating a "bottleneck" pattern: expand to a larger space, apply nonlinearity, then project back.

**Note:** We pass `dropout_rate` as a constructor argument rather than using a global variable, following standard PyTorch conventions.

**Hint:** Use `nn.Sequential` to chain the layers: `self.net = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.ReLU(), nn.Linear(4*n_embd, n_embd), nn.Dropout(dropout_rate))`. ([nn.Sequential docs](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html))

In [ ]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network.

    Two-layer MLP with ReLU activation and dropout.
    Inner dimension is 4 * n_embd.
    """

    def __init__(self, n_embd, dropout_rate):
        """Initialize the feed-forward network.

        Args:
            n_embd: Input/output dimension.
            dropout_rate: Dropout probability.

        Create self.net as an nn.Sequential with:
            1. nn.Linear(n_embd, 4 * n_embd)
            2. nn.ReLU()
            3. nn.Linear(4 * n_embd, n_embd)
            4. nn.Dropout(dropout_rate)
        """
        super().__init__()
        ## YOUR CODE HERE
        pass

    def forward(self, x):
        """Forward pass.

        Args:
            x: Tensor of shape (B, T, n_embd)

        Returns:
            Tensor of shape (B, T, n_embd)
        """
        ## YOUR CODE HERE
        pass

In [ ]:
# Sanity checks for FeedForward

test_ff = FeedForward(n_embd, dropout_rate)
test_ff.eval()

x_test = torch.randn(2, 8, n_embd)
with torch.no_grad():
    out_test = test_ff(x_test)

assert out_test.shape == x_test.shape, \
    f"FFN output shape {out_test.shape} should match input shape {x_test.shape}"

# Check that the inner dimension is 4 * n_embd
first_linear = test_ff.net[0]
assert first_linear.out_features == 4 * n_embd, \
    f"Inner dimension should be {4 * n_embd}, got {first_linear.out_features}"

print(f"FFN input shape:  {x_test.shape}")
print(f"FFN output shape: {out_test.shape}")
print(f"Inner dimension:  {first_linear.out_features}")
print("\nFeedForward sanity checks passed!")

### 3.2 Transformer Block (10 points)

Assemble a complete transformer block using the pre-norm architecture:

```
x ──┬── LayerNorm ── MultiHeadAttention ──┐
    └─────────────────────────────────── + ──┬── LayerNorm ── FeedForward ──┐
                                             └──────────────────────────── + ── output
```

The **residual connections** (the `+` operations) are critical. They allow gradients to flow directly through the network, making deep transformers trainable. Conceptually, each sublayer learns a *refinement* that gets added to the main representation, rather than completely replacing it.

See Jurafsky & Martin, Section 8.2, for the "residual stream" interpretation: the input flows along a main path, with each sublayer branching off, computing a delta, and adding it back.

**Pre-norm vs. Post-norm:** We apply LayerNorm *before* each sublayer (pre-norm), which is standard in modern transformers (GPT-2 and later). The original transformer paper used post-norm (LayerNorm after the residual addition), which can be harder to train.

In [ ]:
class TransformerBlock(nn.Module):
    """A single transformer block: attention + feedforward with residual connections.

    Uses pre-norm architecture: LayerNorm is applied before each sublayer.
    """

    def __init__(self, n_embd, n_head, dropout_rate):
        """Initialize a transformer block.

        Args:
            n_embd: Embedding dimension.
            n_head: Number of attention heads.
            dropout_rate: Dropout probability.

        Create:
            self.ln1: nn.LayerNorm(n_embd) -- before attention
            self.attn: MultiHeadAttention(n_embd, n_head, dropout_rate)
            self.ln2: nn.LayerNorm(n_embd) -- before feedforward
            self.ff: FeedForward(n_embd, dropout_rate)
        """
        super().__init__()
        ## YOUR CODE HERE
        pass

    def forward(self, x):
        """Forward pass with residual connections.

        Args:
            x: Input tensor of shape (B, T, n_embd)

        Returns:
            Output tensor of shape (B, T, n_embd)

        Computation (pre-norm with residual connections):
            x = x + self.attn(self.ln1(x))
            x = x + self.ff(self.ln2(x))
            return x
        """
        ## YOUR CODE HERE
        pass

In [ ]:
# Sanity checks for TransformerBlock

test_block = TransformerBlock(n_embd, n_head, dropout_rate)
test_block.eval()

B_test, T_test = 2, 8
x_test = torch.randn(B_test, T_test, n_embd)
with torch.no_grad():
    out_test = test_block(x_test)

assert out_test.shape == x_test.shape, \
    f"Block output shape {out_test.shape} should match input shape {x_test.shape}"

# Verify the block decomposes correctly (testing residual connections)
with torch.no_grad():
    ln1_out = test_block.ln1(x_test)
    attn_out = test_block.attn(ln1_out)
    after_attn = x_test + attn_out

    ln2_out = test_block.ln2(after_attn)
    ff_out = test_block.ff(ln2_out)
    expected = after_attn + ff_out

assert torch.allclose(out_test, expected, atol=1e-5), \
    "Block output should equal x + attn(ln1(x)) + ff(ln2(x + attn(ln1(x))))"

print(f"Block input shape:  {x_test.shape}")
print(f"Block output shape: {out_test.shape}")
print("Residual connection decomposition matches.")
print("\nTransformerBlock sanity checks passed!")

### 3.3 Residual Connection Verification (5 points) [MANUALLY GRADED]

Write a test that empirically verifies the residual connections are working correctly. The key insight: if the sublayer outputs are near zero, the block output should approximately equal the input.

**Why this matters:** A common bug is forgetting to add the residual connection (writing `x = self.attn(self.ln1(x))` instead of `x = x + self.attn(self.ln1(x))`). This test catches that bug.

**Your test should:**
1. Create a `TransformerBlock` and set it to eval mode (`block.eval()` — this disables dropout so the output is deterministic). ([model.eval() docs](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.eval))
2. Manually set all weights in the attention and feedforward sublayers to near-zero (so their outputs are approximately zero).
3. Pass a random input through the block.
4. Assert that the output is approximately equal to the input (within tolerance).

**Note:** Use the global variables `n_embd`, `n_head`, and `dropout_rate` (defined in Section 0.2) when creating the `TransformerBlock`.

In [ ]:
def test_residual_connections():
    """Test that residual connections pass through the input when sublayers output ~zero.

    Steps:
        1. Create a TransformerBlock and set to eval mode
        2. Zero out all parameters in self.attn and self.ff so their outputs are ~0
           (Hint: iterate over block.attn.parameters() and block.ff.parameters(),
            and set param.data to zeros using param.data.zero_())
        3. Create a random input x of shape (1, 8, n_embd)
        4. Run the block: output = block(x)
        5. Assert torch.allclose(output, x, atol=1e-4)
        6. Print a success message

    If this test fails, the residual connections are not implemented correctly.
    """
    ## YOUR CODE HERE
    pass

test_residual_connections()

---
## Section 4: Full Model, Training & Generation (23 points)

Now you will assemble all components into a complete decoder-only transformer language model, train it on Tiny Shakespeare, and generate text.

The full model architecture:
1. **Token embedding:** Maps each token ID to a dense vector of size `n_embd`
2. **Position embedding:** Adds learned position information (one vector per position, up to `block_size`)
3. **Transformer blocks:** Stack of `n_layer` TransformerBlocks
4. **Final LayerNorm:** Applied after all blocks
5. **Language model head:** Linear projection from `n_embd` to `vocab_size` (produces logits for next-token prediction)

**Lectures:** M06.1.6--M06.1.7, M06.2.1--M06.2.3 | **Textbook:** Chapter 8, Sections 8.4--8.6

### 4.1 Transformer Language Model (10 points)

Implement the full model. The forward pass:
1. Look up token embeddings and position embeddings, add them together.
2. Pass through the stack of transformer blocks.
3. Apply final layer normalization.
4. Project to vocabulary size to get logits.
5. If targets are provided, compute cross-entropy loss.

**Important:** Store `block_size` as `self.block_size` — `generate()` uses `model.block_size` to crop the context. Also store `self.vocab_size = vocab_size` for reference.

In [ ]:
class TransformerLM(nn.Module):
    """Decoder-only transformer language model.

    Combines token embeddings, position embeddings, transformer blocks,
    and a language model head for next-token prediction.
    """

    def __init__(self, vocab_size, n_embd, block_size, n_head, n_layer, dropout_rate):
        """Initialize the transformer language model.

        Args:
            vocab_size: Number of tokens in the vocabulary.
            n_embd: Embedding dimension.
            block_size: Maximum sequence length (context window).
            n_head: Number of attention heads per block.
            n_layer: Number of transformer blocks to stack.
            dropout_rate: Dropout probability.

        Create:
            self.token_emb: nn.Embedding(vocab_size, n_embd)
            self.pos_emb:   nn.Embedding(block_size, n_embd)
            self.blocks:    nn.Sequential of n_layer TransformerBlocks
            self.ln_f:      nn.LayerNorm(n_embd)  -- final layer norm
            self.lm_head:   nn.Linear(n_embd, vocab_size)  -- logit projection

        Also store:
            self.vocab_size = vocab_size  (for reference)
            self.block_size = block_size  (used by generate() to crop context)
        """
        super().__init__()
        ## YOUR CODE HERE
        pass

    def forward(self, idx, targets=None):
        """Forward pass: compute logits and optionally the loss.

        Args:
            idx: Token indices of shape (B, T), where B = batch size, T = seq length.
            targets: Optional target token indices of shape (B, T).

        Returns:
            logits: Tensor of shape (B, T, vocab_size) -- predicted token scores.
            loss: Cross-entropy loss (scalar) if targets provided, else None.

        Steps:
            1. Look up token embeddings: self.token_emb(idx)  -> (B, T, n_embd)
            2. Create position indices: torch.arange(T, device=idx.device)
            3. Look up position embeddings: self.pos_emb(positions)  -> (T, n_embd)
            4. Add token + position embeddings  -> (B, T, n_embd)
            5. Pass through self.blocks  -> (B, T, n_embd)
            6. Apply self.ln_f  -> (B, T, n_embd)
            7. Apply self.lm_head  -> (B, T, vocab_size)
            8. If targets is not None:
               - Reshape logits to (B*T, vocab_size) and targets to (B*T,)
               - Compute F.cross_entropy(logits, targets)
            9. Return logits (reshaped back if needed) and loss
        """
        B, T = idx.shape
        ## YOUR CODE HERE
        pass

In [ ]:
# Sanity checks for TransformerLM

test_model = TransformerLM(vocab_size, n_embd, block_size, n_head, n_layer, dropout_rate).to(device)
test_model.eval()

# Check embedding layers
assert test_model.token_emb.num_embeddings == vocab_size, "Token embedding should have vocab_size entries"
assert test_model.token_emb.embedding_dim == n_embd, "Token embedding dim should be n_embd"
assert test_model.pos_emb.num_embeddings == block_size, "Position embedding should have block_size entries"
assert test_model.pos_emb.embedding_dim == n_embd, "Position embedding dim should be n_embd"

# Check forward pass without targets
B_test, T_test = 2, 16
idx_test = torch.randint(0, vocab_size, (B_test, T_test), device=device)
with torch.no_grad():
    logits_test, loss_test = test_model(idx_test)

assert logits_test.shape == (B_test, T_test, vocab_size), \
    f"Logits shape should be ({B_test}, {T_test}, {vocab_size}), got {logits_test.shape}"
assert loss_test is None, "Loss should be None when targets are not provided"

# Check forward pass with targets
targets_test = torch.randint(0, vocab_size, (B_test, T_test), device=device)
with torch.no_grad():
    logits_test2, loss_test2 = test_model(idx_test, targets_test)

assert loss_test2 is not None, "Loss should not be None when targets are provided"
assert loss_test2.dim() == 0, f"Loss should be a scalar, got shape {loss_test2.shape}"
# With random weights, loss should be approximately -log(1/vocab_size) = log(vocab_size)
expected_loss = np.log(vocab_size)
assert abs(loss_test2.item() - expected_loss) < 1.0, \
    f"Initial loss should be near {expected_loss:.2f} (random guessing), got {loss_test2.item():.2f}"

# Count parameters
n_params = sum(p.numel() for p in test_model.parameters())
print(f"TransformerLM architecture:")
print(f"  vocab_size = {vocab_size}")
print(f"  n_embd     = {n_embd}")
print(f"  block_size = {block_size}")
print(f"  n_head     = {n_head}")
print(f"  n_layer    = {n_layer}")
print(f"  Parameters = {n_params:,}")
print(f"\nLogits shape (no targets): {logits_test.shape}")
print(f"Loss (with targets):       {loss_test2.item():.4f}")
print(f"Expected initial loss:     ~{expected_loss:.4f} (random guessing)")
print("\nTransformerLM sanity checks passed!")

### 4.2 Training Loop (5 points) [MANUALLY GRADED]

Implement the training loop. Use the AdamW optimizer and train for 5000 iterations, printing training and validation loss every 500 steps.

**Expected behavior:**
- Initial loss: ~4.2 (random guessing over 65 characters: $\ln(65) \approx 4.17$)
- After training: loss should drop to ~1.5--1.8
- Training time: ~15-30 minutes on CPU, ~2-3 minutes on a free Colab GPU

> **Tip: switch Colab to a GPU runtime if you can.** In Colab, go to *Runtime -> Change runtime type -> T4 GPU (free)*. Training is roughly 10x faster on GPU. The model is small enough that CPU training works, but you will save a lot of time.

**Optimizer:** Use `torch.optim.AdamW(model.parameters(), lr=learning_rate)`. The global `learning_rate` is set in Section 0.2. ([AdamW docs](https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html))

**Gradient update pattern** (one iteration):
```python
optimizer.zero_grad(set_to_none=True)  # clear accumulated gradients
loss.backward()                         # compute gradients
optimizer.step()                        # update weights
```

**Recording losses:** Store `loss.item()` (a Python float) rather than the tensor itself, so the loss list stays on CPU and works with matplotlib.

In [ ]:
def train_model(model, n_iters=5000, eval_interval=500):
    """Train the transformer language model.

    Args:
        model: A TransformerLM model (already on the correct device).
        n_iters: Number of training iterations.
        eval_interval: Print train/val loss every this many steps.

    Returns:
        train_losses: List of training losses recorded at each eval_interval.
        val_losses: List of validation losses recorded at each eval_interval.

    Steps:
        1. Create an AdamW optimizer: `torch.optim.AdamW(model.parameters(), lr=learning_rate)`
           (uses the global `learning_rate` from Section 0.2)
        2. For each iteration:
           a. Sample a batch from the training set using get_batch('train').
           b. Forward pass: compute logits and loss.
           c. `optimizer.zero_grad(set_to_none=True)` → `loss.backward()` → `optimizer.step()`
           d. Every eval_interval steps:
              - Evaluate on a validation batch (with torch.no_grad())
              - Record `loss.item()` (a float) to train_losses / val_losses
              - Print step, train loss, and val loss
    """
    ## YOUR CODE HERE
    pass

In [ ]:
# Skip when imported as a module (slow training loop)
if __name__ == '__main__':
    # Create and train the model
    model = TransformerLM(vocab_size, n_embd, block_size, n_head, n_layer, dropout_rate).to(device)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {n_params:,}")
    print(f"Training for 5000 iterations...\n")

    start_time = time.time()
    train_losses, val_losses = train_model(model, n_iters=5000, eval_interval=500)
    elapsed = time.time() - start_time

    print(f"\nTraining complete in {elapsed / 60:.2f} minutes.")
    print(f"Final train loss: {train_losses[-1]:.4f}")
    print(f"Final val loss:   {val_losses[-1]:.4f}")

In [ ]:
# Skip when imported as a module (needs trained model)
if __name__ == '__main__':
    # Plot training and validation loss
    plt.figure(figsize=(10, 4))
    steps = [i * 500 for i in range(len(train_losses))]
    # Use .item() in case losses were stored as tensors rather than floats
    train_vals = [l.item() if hasattr(l, 'item') else l for l in train_losses]
    val_vals   = [l.item() if hasattr(l, 'item') else l for l in val_losses]
    plt.plot(steps, train_vals, label='Train Loss', alpha=0.8)
    plt.plot(steps, val_vals,   label='Val Loss', alpha=0.8)
    plt.xlabel('Training Iterations')
    plt.ylabel('Cross-Entropy Loss')
    plt.title('Transformer Language Model Training')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### 4.3 Text Generation (5 points)

Implement autoregressive text generation. The model generates one token at a time:
1. Feed the current context into the model to get logits for the next token.
2. Apply temperature scaling to the logits (divide by temperature before softmax).
3. Sample from the resulting probability distribution.
4. Append the sampled token to the context and repeat.

**Temperature** controls the randomness of generation:
- $T = 1.0$: Standard sampling (default)
- $T < 1.0$: More deterministic (sharper distribution, more likely to pick the top token)
- $T > 1.0$: More random (flatter distribution, more diverse but less coherent)

**Important:** The context must be cropped to `block_size` tokens.
Use: `idx_cond = idx[:, -model.block_size:]`  (takes the **last** `block_size` tokens when the sequence grows longer than the context window), since the model's position embeddings only support positions $0$ to $\text{block\_size} - 1$.

In [ ]:
def generate(model, prompt, max_new_tokens=500, temperature=1.0):
    """Generate text autoregressively from a prompt.

    Args:
        model: A trained TransformerLM model.
        prompt: A string to use as the starting context.
        max_new_tokens: Number of new tokens to generate.
        temperature: Sampling temperature (default 1.0).

    Returns:
        A string containing the prompt followed by the generated text.

    Steps:
        1. Encode the prompt to a tensor of shape (1, len(prompt)) on the correct device.
           Use: torch.tensor([encode(prompt)], device=device)
        2. Set model to eval mode and use torch.no_grad().
        3. For each new token:
           a. Crop: `idx_cond = idx[:, -model.block_size:]`  (keeps the last block_size tokens)
           b. Forward pass to get logits of shape (1, T, vocab_size).
           c. Take logits for the last position only: logits[:, -1, :]
           d. Divide logits by temperature.
           e. Apply softmax to get probabilities.
           f. Sample next token using torch.multinomial(probs, num_samples=1).
           g. Append the sampled token to the context.
        4. Decode the full sequence (prompt + generated) back to a string.
    """
    ## YOUR CODE HERE
    pass

In [ ]:
# Skip when imported as a module (needs trained model)
if __name__ == '__main__':
    # Sanity check: generate should produce a string
    test_output = generate(model, "ROMEO:", max_new_tokens=10, temperature=1.0)
    assert isinstance(test_output, str), f"generate should return a string, got {type(test_output)}"
    assert test_output.startswith("ROMEO:"), "Output should start with the prompt"
    assert len(test_output) > len("ROMEO:"), "Output should be longer than the prompt"

    print("Generation sanity check passed!\n")
    print("="*60)
    print("Generated text (temperature=1.0):")
    print("="*60)
    print(generate(model, "ROMEO:\nWherefore art thou?", max_new_tokens=300, temperature=1.0))

In [ ]:
# Skip when imported as a module (needs trained model)
if __name__ == '__main__':
    # Compare different temperatures
    prompt = "To be, or not to be,"

    for temp in [0.5, 1.0, 1.5]:
        print(f"\n{'='*60}")
        print(f"Temperature = {temp}")
        print(f"{'='*60}")
        print(generate(model, prompt, max_new_tokens=200, temperature=temp))

### 4.4 Generation Failure Analysis (3 points) [WRITEUP]

Your model generates text that is recognizably Shakespearean in style, but it's far from perfect. Examine a longer sample (generate at least **300 characters of new output** — not counting the prompt) and identify specific failures.

**[WRITEUP] Answer 4.4:**

Generate at least 300 characters of text with temperature=1.0 and identify **3 specific failures** from the categories below. For each, quote the relevant passage and explain why your model produces this error:

1. **Repetition:** Does the model get stuck repeating a phrase or pattern? Why might a small model with limited context do this?
2. **Nonsense words:** Does it generate character sequences that aren't real English words? Why does character-level tokenization make this more likely than word-level?
3. **Grammatical errors:** Does it start a sentence or speech prefix correctly but lose coherence? Connect to the model's limited context window (`block_size=64` characters ≈ 10-12 words).
4. **Inconsistent character names:** Does it mix up Shakespeare characters or invent new ones? Why?

*(This exercise connects to Lecture M06.3.1 on scaling: many of these failures would improve with a larger model, more data, or subword tokenization.)*

---
## Section 5: RNN vs Transformer Analysis (10 points) [ALL WRITEUP]

Answer the following questions in your `report.pdf`. Use the code cells provided to generate any figures or results you need.


### [WRITEUP] Answer 5.1 (5 points): Attention Patterns

After training, extract and visualize the attention weights from your model on a sample input.

Use the code cell below to:
1. Run a forward pass on a sample string (e.g., `"To be, or not to be,"`).
2. Extract the attention weights from at least **4 different heads** (across 2+ layers).
3. Plot heatmaps of the attention weight matrices using `matplotlib.imshow`.

In your report, include the heatmaps and answer:

**(a)** Look specifically for these common attention patterns:
- **Positional/local attention:** Does any head consistently attend to the immediately preceding token (a diagonal stripe one position to the left)? What linguistic function might this serve?
- **Content-based attention:** Does any head attend to specific characters like spaces, newlines, or punctuation regardless of position?
- **Diffuse attention:** Does any head spread attention roughly uniformly across all previous positions? What role might this play in the model?

**(b)** Do different heads within the same layer show different patterns? What about the same head position across different layers (e.g., head 0 of layer 0 vs head 0 of layer 3)?

**(c)** Research by Olsson et al. (2022) identified "induction heads" — pairs of attention heads that implement a simple copying pattern: one head attends to the previous occurrence of a token, and another head copies what came after it. Do you see any evidence of this pattern in your model's attention weights?

**Hint:** After a forward pass, access weights via `model.blocks[layer_idx].attn.heads[head_idx].attn_weights`. Shape: `(1, T, T)` where T = len(sample_text).

In [ ]:
# Skip when imported as a module (needs trained model)
if __name__ == '__main__':
    # Use this cell to extract and visualize attention patterns.

    sample_text = "To be, or not to be,"
    sample_tokens = torch.tensor([encode(sample_text)], device=device)

    # Forward pass to populate attention weights
    model.eval()
    with torch.no_grad():
        _ = model(sample_tokens)

    # Example: access attention weights from layer 0, head 0
    attn_weights_00 = model.blocks[0].attn.heads[0].attn_weights[0].cpu().numpy()
    # Shape: (T, T) where T = len(sample_text)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(attn_weights_00, cmap='viridis', aspect='auto')
    ax.set_xticks(range(len(sample_text))); ax.set_xticklabels(list(sample_text), fontsize=8)
    ax.set_yticks(range(len(sample_text))); ax.set_yticklabels(list(sample_text), fontsize=8)
    ax.set_title('Layer 0, Head 0'); plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()
    print("Add 3 more head visualizations below:")

    ## YOUR CODE HERE
    # Plot at least 4 different heads total (across at least 2 layers).
    # The example below plots layer 0, head 0 — complete 3 more on your own.
    # 1. Extract attention weights from at least 2 heads
    # 2. Plot heatmaps using plt.imshow
    #    - Use the characters of sample_text as tick labels
    #    - Add a colorbar
    #    - Label which layer/head each plot shows

### [WRITEUP] Answer 5.2 (5 points): From RNNs to Transformers

You trained an RNN baseline (Section 1.3) and a Transformer (Section 4) on the **same** Tiny Shakespeare next-character prediction task. Compare your two models on the following dimensions:

**(a) Validation loss.** Report the final validation loss of your RNN baseline (after 1000 iterations) and your Transformer (after full training). Which performs better, and by how much?

**(b) Architectural reasons.** RNNs have two well-known limitations:

1. **Sequential processing:** Each time step depends on the previous one, so RNNs cannot be parallelized across time steps. Training is slow.
2. **Long-range information bottleneck:** Information from early time steps must squeeze through the recurrent hidden state, where it can be diluted or lost.

How does the Transformer architecture address each of these limitations? Be specific: which components of the Transformer (e.g., self-attention, position embeddings, parallel computation over the sequence) map onto which RNN limitation?

**(c) Tradeoffs.** Are there any aspects in which RNNs still have an advantage over Transformers? Think about memory, parameter count, or sequence-length scaling.


---
## Section 6: Mathematical Analysis (7 points) [WRITEUP]

Answer the question below in your `hw05_writeup.pdf`.


### [WRITEUP] Answer 6.1 (7 points): Why Scaled Dot-Product?

The attention formula divides the dot product $QK^\top$ by $\sqrt{d_k}$:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

**(a)** What would happen if you removed this scaling factor? Specifically, what would happen to the softmax output as $d_k$ grows? (Hint: think about what softmax does when its inputs have very large magnitude.)

**(b)** Why does the magnitude of $QK^\top$ grow with $d_k$?  
*(Variance reminder: if $X$ and $Y$ are independent with variances $\sigma^2_X$ and $\sigma^2_Y$, then $\text{Var}(X+Y) = \sigma^2_X + \sigma^2_Y$; and $\text{Var}(cX) = c^2 \sigma^2_X$.)* Assume the entries of $Q$ and $K$ are independent random variables with mean 0 and variance 1. What is the variance of a single element of $QK^\top$? (Hint: the dot product of two $d_k$-dimensional random vectors is a sum of $d_k$ independent terms.)

**(c)** Why does dividing by $\sqrt{d_k}$ fix this problem? What is the variance after scaling?

**Reference:** Vaswani et al. (2017), Section 3.2.1; Lecture M06.1.3; Textbook Chapter 8, Section 8.3.

---
## Submission Checklist

Before submitting, verify:

- [ ] All code cells run without errors (Kernel -> Restart & Run All)
- [ ] All assertion/sanity checks pass
- [ ] RNN baseline trains to a reasonable loss (final val loss < 3.0 after 1000 steps)
- [ ] Transformer trains to a reasonable loss (< 2.0)
- [ ] Generated text is recognizably Shakespearean in style
- [ ] `hw05_writeup.pdf` contains answers to all **[WRITEUP]** questions:
  - **Answer 4.4:** Generation failure analysis (3 specific failures with explanations)
  - **Answer 5.1:** Attention pattern visualization (4+ heads, positional/content/diffuse patterns)
  - **Answer 5.2:** RNN vs Transformer comparison (val loss numbers + architectural reasoning + tradeoffs)
  - **Answer 6.1:** Scaled dot-product attention explanation

- [ ] Export your notebook as `hw05.py`: *File → Download → Download .py* (or `jupyter nbconvert --to script hw05_notebook.ipynb --output hw05`)
- [ ] Files are named exactly: `hw05.py`, `hw05.ipynb`, and `hw05_writeup.pdf`
